In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/application_train.csv')
df.shape
df.head()
df['TARGET'].value_counts(normalize=True)  # cek imbalance-nya

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

In [ ]:
df.dtypes.value_counts()  # berapa kolom numerik vs kategorikal

float64    65
int64      41
object     16
Name: count, dtype: int64

In [3]:
df.shape


(307511, 122)

In [4]:
missing = df.isnull().mean().sort_values(ascending=False)
missing[missing > 0].head(30)

COMMONAREA_MEDI             0.698723
COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
NONLIVINGAPARTMENTS_MEDI    0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_MODE       0.683550
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
FLOORSMIN_AVG               0.678486
FLOORSMIN_MODE              0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_MEDI            0.664978
YEARS_BUILD_MODE            0.664978
YEARS_BUILD_AVG             0.664978
OWN_CAR_AGE                 0.659908
LANDAREA_MEDI               0.593767
LANDAREA_MODE               0.593767
LANDAREA_AVG                0.593767
BASEMENTAREA_MEDI           0.585160
BASEMENTAREA_AVG            0.585160
BASEMENTAREA_MODE           0.585160
EXT_SOURCE_1                0.563811
NONLIVINGAREA_MODE          0.551792
NONLIVINGAREA_AVG           0.551792
NONLIVINGAREA_MEDI          0.551792
E

In [5]:
missing[(missing > 0) & (missing < 0.55)]

ELEVATORS_MEDI                  0.532960
ELEVATORS_AVG                   0.532960
ELEVATORS_MODE                  0.532960
WALLSMATERIAL_MODE              0.508408
APARTMENTS_MEDI                 0.507497
APARTMENTS_AVG                  0.507497
APARTMENTS_MODE                 0.507497
ENTRANCES_MEDI                  0.503488
ENTRANCES_AVG                   0.503488
ENTRANCES_MODE                  0.503488
LIVINGAREA_AVG                  0.501933
LIVINGAREA_MODE                 0.501933
LIVINGAREA_MEDI                 0.501933
HOUSETYPE_MODE                  0.501761
FLOORSMAX_MODE                  0.497608
FLOORSMAX_MEDI                  0.497608
FLOORSMAX_AVG                   0.497608
YEARS_BEGINEXPLUATATION_MODE    0.487810
YEARS_BEGINEXPLUATATION_MEDI    0.487810
YEARS_BEGINEXPLUATATION_AVG     0.487810
TOTALAREA_MODE                  0.482685
EMERGENCYSTATE_MODE             0.473983
OCCUPATION_TYPE                 0.313455
EXT_SOURCE_3                    0.198253
AMT_REQ_CREDIT_B

In [6]:
df.groupby('FLAG_OWN_CAR')['OWN_CAR_AGE'].apply(lambda x: x.isnull().mean())

FLAG_OWN_CAR
N    1.000000
Y    0.000048
Name: OWN_CAR_AGE, dtype: float64

In [7]:
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    print(col)
    print(df.groupby('TARGET')[col].mean())
    print('---')

EXT_SOURCE_1
TARGET
0    0.511461
1    0.386968
Name: EXT_SOURCE_1, dtype: float64
---
EXT_SOURCE_2
TARGET
0    0.523479
1    0.410935
Name: EXT_SOURCE_2, dtype: float64
---
EXT_SOURCE_3
TARGET
0    0.520969
1    0.390717
Name: EXT_SOURCE_3, dtype: float64
---


In [8]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()['TARGET'].sort_values()
print(correlations.head(10))   # paling negatif (makin tinggi nilai, makin ga default)
print(correlations.tail(11))   # paling positif (makin tinggi nilai, makin default) - 11 karena TARGET sendiri ikut

EXT_SOURCE_3                 -0.178919
EXT_SOURCE_2                 -0.160472
EXT_SOURCE_1                 -0.155317
DAYS_EMPLOYED                -0.044932
FLOORSMAX_AVG                -0.044003
FLOORSMAX_MEDI               -0.043768
FLOORSMAX_MODE               -0.043226
AMT_GOODS_PRICE              -0.039645
REGION_POPULATION_RELATIVE   -0.037227
ELEVATORS_AVG                -0.034199
Name: TARGET, dtype: float64
DAYS_REGISTRATION              0.041975
FLAG_DOCUMENT_3                0.044346
REG_CITY_NOT_LIVE_CITY         0.044395
FLAG_EMP_PHONE                 0.045982
REG_CITY_NOT_WORK_CITY         0.050994
DAYS_ID_PUBLISH                0.051457
DAYS_LAST_PHONE_CHANGE         0.055218
REGION_RATING_CLIENT           0.058899
REGION_RATING_CLIENT_W_CITY    0.060893
DAYS_BIRTH                     0.078239
TARGET                         1.000000
Name: TARGET, dtype: float64


In [9]:
print(df['DAYS_EMPLOYED'].describe())
print((df['DAYS_EMPLOYED'] == 365243).sum())

count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64
55374


In [15]:
# reload fresh
df = pd.read_csv('../data/raw/application_train.csv')

# 1. bikin flag dulu SEBELUM replace
df['DAYS_EMPLOYED_ANOMALY'] = df['DAYS_EMPLOYED'] == 365243

# 2. baru replace
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# 3. cek
print(df['DAYS_EMPLOYED_ANOMALY'].value_counts())
print(df.groupby('DAYS_EMPLOYED_ANOMALY')['TARGET'].mean())

DAYS_EMPLOYED_ANOMALY
False    252137
True      55374
Name: count, dtype: int64
DAYS_EMPLOYED_ANOMALY
False    0.086600
True     0.053996
Name: TARGET, dtype: float64


In [16]:
# 1. Liat gambaran umum angka-angka keuangan
print(df[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']].describe())

# 2. Hitung rasio total utang terhadap gaji & liat rata-ratanya per kelompok TARGET
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
print(df.groupby('TARGET')['CREDIT_INCOME_RATIO'].mean())

# 3. Hitung rasio cicilan bulanan terhadap gaji & liat rata-ratanya per kelompok TARGET
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
print(df.groupby('TARGET')['ANNUITY_INCOME_RATIO'].mean())

       AMT_INCOME_TOTAL    AMT_CREDIT    AMT_ANNUITY  AMT_GOODS_PRICE
count      3.075110e+05  3.075110e+05  307499.000000     3.072330e+05
mean       1.687979e+05  5.990260e+05   27108.573909     5.383962e+05
std        2.371231e+05  4.024908e+05   14493.737315     3.694465e+05
min        2.565000e+04  4.500000e+04    1615.500000     4.050000e+04
25%        1.125000e+05  2.700000e+05   16524.000000     2.385000e+05
50%        1.471500e+05  5.135310e+05   24903.000000     4.500000e+05
75%        2.025000e+05  8.086500e+05   34596.000000     6.795000e+05
max        1.170000e+08  4.050000e+06  258025.500000     4.050000e+06
TARGET
0    3.963729
1    3.887438
Name: CREDIT_INCOME_RATIO, dtype: float64
TARGET
0    0.180530
1    0.185482
Name: ANNUITY_INCOME_RATIO, dtype: float64


In [17]:
import sys
sys.path.append('../src')
from data_cleaning import clean_application_data

df_raw = pd.read_csv('../data/raw/application_train.csv')
df_clean = clean_application_data(df_raw)

print(df_clean.shape)
print(df_clean.isnull().mean().sort_values(ascending=False).head(10))

(307511, 79)
DAYS_EMPLOYED       0.180072
SK_ID_CURR          0.000000
FLAG_DOCUMENT_12    0.000000
FLAG_DOCUMENT_10    0.000000
FLAG_DOCUMENT_9     0.000000
FLAG_DOCUMENT_8     0.000000
FLAG_DOCUMENT_7     0.000000
FLAG_DOCUMENT_6     0.000000
FLAG_DOCUMENT_5     0.000000
FLAG_DOCUMENT_4     0.000000
dtype: float64


In [18]:
import importlib
import data_cleaning
importlib.reload(data_cleaning)  # reload module biar perubahan kepake
from data_cleaning import clean_application_data

df_raw = pd.read_csv('../data/raw/application_train.csv')
df_clean = clean_application_data(df_raw)

print(df_clean.shape)
print(df_clean.isnull().sum().sum())  # total semua missing value, harusnya 0

(307511, 79)
0


In [19]:
df_clean.to_csv('../data/processed/application_train_clean.csv', index=False)